In [1]:
import os
import subprocess
import torch

print("=" * 60)
print("PYTORCH CUDA CHECK")
print("=" * 60)

# PyTorch info
print("Torch version :", torch.__version__)
print("CUDA compiled :", torch.version.cuda)

# CUDA availability
print("CUDA available:", torch.cuda.is_available())

try:
    print("Device count  :", torch.cuda.device_count())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i} Name   :", torch.cuda.get_device_name(i))

        # actual GPU computation test
        x = torch.randn(5000, 5000, device="cuda")
        y = torch.mm(x, x)

        print("GPU compute test: SUCCESS")
        print("Tensor device   :", y.device)

    else:
        print("GPU NOT AVAILABLE")

except Exception as e:
    print("CUDA ERROR:")
    print(e)

print("\n" + "=" * 60)
print("NVIDIA-SMI CHECK")
print("=" * 60)

try:
    result = subprocess.check_output(["nvidia-smi"]).decode()
    print(result)
except Exception as e:
    print("nvidia-smi failed:")
    print(e)

PYTORCH CUDA CHECK
Torch version : 2.5.1+cu121
CUDA compiled : 12.1
CUDA available: True
Device count  : 8
GPU 0 Name   : NVIDIA A100-SXM4-80GB
GPU 1 Name   : NVIDIA A100-SXM4-80GB
GPU 2 Name   : NVIDIA A100-SXM4-80GB
GPU 3 Name   : NVIDIA A100-SXM4-80GB
GPU 4 Name   : NVIDIA A100-SXM4-80GB
GPU 5 Name   : NVIDIA A100-SXM4-80GB
GPU 6 Name   : NVIDIA A100-SXM4-80GB
GPU 7 Name   : NVIDIA A100-SXM4-80GB
GPU compute test: SUCCESS
Tensor device   : cuda:0

NVIDIA-SMI CHECK
Wed Jun 10 17:12:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                   

In [ ]:
!lscpu

In [2]:
import multiprocessing
print(multiprocessing.cpu_count())

96


In [3]:
import psutil
print(psutil.virtual_memory())

svmem(total=1902382264320, available=1821722730496, percent=4.2, used=80659533824, free=230225510400, active=211709685760, inactive=1422933106688, buffers=1739612160, cached=1604538781696, shared=5629947904, slab=18312974336)


In [1]:
!uptime

 01:14:20 up 77 days, 17:06, 18 users,  load average: 143.88, 132.00, 136.60


In [1]:
import os

print("rclone config exists:",
      os.path.exists("/home/nvidia/.config/rclone/rclone.conf"))

print("ADHD200 exists:",
      os.path.exists("/mnt/ADHD200"))

print("FC matrices:",
      os.path.exists("/mnt/ADHD200/03_fc_matrices/X_fc.npy"))

rclone config exists: False
ADHD200 exists: True
FC matrices: True


In [2]:
import numpy as np
import pandas as pd

X_fc = np.load("/mnt/ADHD200/03_fc_matrices/X_fc.npy")
y = np.load("/mnt/ADHD200/03_fc_matrices/y_binary.npy")
sites = np.load("/mnt/ADHD200/03_fc_matrices/sites.npy")

print("X_fc:", X_fc.shape)
print("y:", y.shape)
print("sites:", len(np.unique(sites)))
print("RAM footprint (GB):", X_fc.nbytes / 1024**3)

X_fc: (497, 190, 190)
y: (497,)
sites: 7
RAM footprint (GB): 0.06683804094791412


In [3]:
import numpy as np
import pandas as pd

X_graph = pd.read_parquet(
    "/mnt/ADHD200/04_features/w1_graph_features.parquet"
)

X_pheno = pd.read_parquet(
    "/mnt/ADHD200/04_features/w1_pheno_features.parquet"
)

print("Graph:", X_graph.shape)
print("Pheno:", X_pheno.shape)

Graph: (497, 23)
Pheno: (497, 10)


In [4]:
print("ADHD:", np.sum(y == 1))
print("Control:", np.sum(y == 0))

unique_sites, counts = np.unique(sites, return_counts=True)
for s, c in zip(unique_sites, counts):
    print(s, c)

ADHD: 216
Control: 281
KKI 83
NYU 93
NeuroIMAGE 48
OHSU 79
Peking_1 85
Peking_2 67
Peking_3 42


In [8]:
import shutil
from pathlib import Path
RUN_ROOT = Path('/mnt/ADHD200_WORKFLOW1_RUN')
print(f'RUN_ROOT exists: {RUN_ROOT.exists()}')
if RUN_ROOT.exists():
    print('WARNING: If this is a fresh run, you MUST delete RUN_ROOT before proceeding.')
    print('To delete, run: !rm -rf /mnt/ADHD200_WORKFLOW1_RUN or uncomment the line below:')
    # shutil.rmtree(RUN_ROOT)


RUN_ROOT exists: False


In [11]:
!pip install -q xgboost lightgbm networkx psutil

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import json
import joblib
import scipy.stats as st
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm
import time
import psutil
from sklearn.model_selection import StratifiedKFold, ParameterGrid, cross_val_predict
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.calibration import calibration_curve
try:
    from lightgbm import LGBMClassifier
except ImportError:
    print('LightGBM not installed')
import networkx as nx
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = [12, 6]
DATA_ROOT = Path('/mnt/ADHD200')
RUN_ROOT = Path('/mnt/ADHD200_WORKFLOW1_RUN')
FC_DIR = DATA_ROOT / '03_fc_matrices'
COHORT_DIR = DATA_ROOT / '01_cohort'
FEATURE_DIR = RUN_ROOT / '04_features'
MODEL_DIR = RUN_ROOT / '05_models' / 'classical_ml'
RESULT_DIR = RUN_ROOT / '06_results' / 'workflow1'
REPORT_DIR = RUN_ROOT / '07_reports'
FORCE_REBUILD_FEATURES = True
USE_CHECKPOINTS = True

print('='*80)
print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT :', RUN_ROOT)
print('FC_DIR   :', FC_DIR)
print('COHORT   :', COHORT_DIR)
print('FEATURES :', FEATURE_DIR)
print('MODELS   :', MODEL_DIR)
print('RESULTS  :', RESULT_DIR)
print('REPORTS  :', REPORT_DIR)
print('='*80)

assert (FC_DIR / 'X_fc.npy').exists(), 'Missing X_fc.npy'
assert (FC_DIR / 'X_fc_clean.npy').exists(), 'Missing X_fc_clean.npy'
assert (FC_DIR / 'X_fc_norm.npy').exists(), 'Missing X_fc_norm.npy'
assert (FC_DIR / 'y.npy').exists(), 'Missing y.npy'
assert (FC_DIR / 'y_binary.npy').exists(), 'Missing y_binary.npy'
assert (FC_DIR / 'sites.npy').exists(), 'Missing sites.npy'
assert (FC_DIR / 'subjects.npy').exists(), 'Missing subjects.npy'

for d in [FEATURE_DIR, MODEL_DIR, RESULT_DIR, REPORT_DIR]: d.mkdir(parents=True, exist_ok=True)
for f in ['FC_only', 'Graph_only', 'Phenotype_only', 'Phenotype_NoIQ', 'FC_Phenotype', 'Graph_Phenotype']:
    (MODEL_DIR / f).mkdir(exist_ok=True)
(MODEL_DIR / 'overall_winner').mkdir(exist_ok=True)
AUDIT_STATE = {}
FILE_MANIFEST = []
def log_file(logical_name, path_obj):
    path_str = str(path_obj.absolute())
    FILE_MANIFEST.append({'logical_name': logical_name, 'absolute_path': path_str, 'exists': path_obj.exists()})
    return path_obj

START_TIME = time.time()
def eta(done, total):
    elapsed = time.time() - START_TIME
    if done == 0: return 'unknown'
    remaining = elapsed * (total - done) / done
    h = int(remaining // 3600)
    m = int((remaining % 3600) // 60)
    return f'{h}h {m}m'

def memory_status():
    mem = psutil.virtual_memory()
    print(f'RAM Used: {mem.used/1024**3:.2f} GB / {mem.total/1024**3:.2f} GB')


DATA_ROOT: /mnt/ADHD200
RUN_ROOT : /mnt/ADHD200_WORKFLOW1_RUN
FC_DIR   : /mnt/ADHD200/03_fc_matrices
COHORT   : /mnt/ADHD200/01_cohort
FEATURES : /mnt/ADHD200_WORKFLOW1_RUN/04_features
MODELS   : /mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml
RESULTS  : /mnt/ADHD200_WORKFLOW1_RUN/06_results/workflow1
REPORTS  : /mnt/ADHD200_WORKFLOW1_RUN/07_reports


In [32]:
fc_path = log_file('X_fc', FC_DIR / 'X_fc_norm.npy')
y_path = log_file('y_binary', FC_DIR / 'y_binary.npy')
sites_path = log_file('sites', FC_DIR / 'sites.npy')
subs_path = log_file('subjects', FC_DIR / 'subjects.npy')
X_fc = np.load(fc_path)
y = np.load(y_path)
sites = np.load(sites_path, allow_pickle=True)
subjects = np.load(subs_path, allow_pickle=True)
sites = np.array([str(s) for s in sites])
site_table = pd.DataFrame([{'site': s, 'n': int((sites==s).sum()), 'n_adhd': int((y[sites==s]==1).sum()), 'n_ctrl': int((y[sites==s]==0).sum())} for s in np.unique(sites)])
print(site_table.to_string(index=False))
w1_manifest = {'subjects': len(y), 'sites': list(np.unique(sites)), 'source': '03_fc_matrices', 'feature_dimension': int(X_fc.shape[1] * (X_fc.shape[1]-1) / 2), 'generated': datetime.now().isoformat()}
with open(RESULT_DIR / 'w1_dataset_manifest.json', 'w') as f: json.dump(w1_manifest, f, indent=2)


      site  n  n_adhd  n_ctrl
       KKI 83      22      61
       NYU 93      54      39
NeuroIMAGE 48      25      23
      OHSU 79      37      42
  Peking_1 85      24      61
  Peking_2 67      35      32
  Peking_3 42      19      23


In [16]:
pheno_path = log_file('pheno_features', FEATURE_DIR / 'w1_pheno_features.parquet')
if (not FORCE_REBUILD_FEATURES) and pheno_path.exists(): X_pheno_raw = pd.read_parquet(pheno_path)
else:
    cohort_path = log_file('master_cohort', COHORT_DIR / 'master_cohort.csv')
    cohort = pd.read_csv(cohort_path)
    cohort['subject_id'] = cohort['subject_id'].astype(str)
    print('master_cohort columns:', cohort.columns.tolist())
    pheno = cohort.copy()
    if 'Handedness' in pheno.columns: pheno['Handedness'] = pheno['Handedness'].replace(-999, np.nan)
    X_pheno_raw = pheno.set_index('subject_id').reindex([str(s) for s in subjects]).reset_index()[[c for c in ['Age','Gender','Handedness','VIQ','PIQ','FIQ','FIQ_source','VIQ_missing','PIQ_missing','FIQ_missing'] if c in pheno.columns]].copy()
    X_pheno_raw.replace([999, -999, -1e9], np.nan, inplace=True)
    assert not np.any(X_pheno_raw.values == 999)
    assert not np.any(X_pheno_raw.values == -999)
    assert not np.any(X_pheno_raw.values == -1e9)
    X_pheno_raw.to_parquet(pheno_path, index=False)

# DType Audit
bad_cols = {'y', 'dx', 'adhd', 'adhd_binary', 'target', 'label', 'diagnosis', 'class', 'group', 'outcome'}
for c in X_pheno_raw.columns:
    assert c.lower() not in bad_cols, f'Target variable {c} leaked into phenotype features'
assert all(np.issubdtype(dt, np.number) for dt in X_pheno_raw.dtypes), 'Non-numeric dtype found in Phenotype data'
X_pheno_raw.dtypes.astype(str).reset_index(name='dtype').rename(columns={'index': 'column'}).to_csv(RESULT_DIR / 'w1_pheno_dtype_audit.csv', index=False)

# Sentinel and Integrity Audit
integ_audit = []
for c in X_pheno_raw.columns:
    vec = X_pheno_raw[c].values
    vec_valid = vec[~np.isnan(vec)]
    if len(vec_valid) == 0: continue
    integ_audit.append({'check': f'No sentinel -999 in {c}', 'status': 'PASS' if not np.any(vec_valid == -999) else 'FAIL'})
    integ_audit.append({'check': f'No sentinel 999 in {c}', 'status': 'PASS' if not np.any(vec_valid == 999) else 'FAIL'})
    integ_audit.append({'check': f'No sentinel -1e9 in {c}', 'status': 'PASS' if not np.any(vec_valid == -1e9) else 'FAIL'})
    integ_audit.append({'check': f'All valid values are finite in {c}', 'status': 'PASS' if np.isfinite(vec_valid).all() else 'FAIL'})

if 'Age' in X_pheno_raw.columns: integ_audit.append({'check': 'Age between 4 and 25', 'status': 'PASS' if X_pheno_raw['Age'].dropna().between(4, 26).all() else 'FAIL'})
if 'Gender' in X_pheno_raw.columns: integ_audit.append({'check': 'Gender values subset of expected codes', 'status': 'PASS' if X_pheno_raw['Gender'].dropna().isin([0, 1, 0.0, 1.0]).all() else 'FAIL'})
if 'Handedness' in X_pheno_raw.columns:
    h_vals = set(X_pheno_raw['Handedness'].dropna().unique())
    if h_vals.issubset({0, 1, 2, 3, 0.0, 1.0, 2.0, 3.0}): integ_audit.append({'check': 'Handedness is valid categorical', 'status': 'PASS'})
    else: integ_audit.append({'check': 'Handedness within valid Edinburgh [-100, 100]', 'status': 'PASS' if X_pheno_raw['Handedness'].dropna().between(-100, 100).all() else 'FAIL'})
for iq_col in ['FIQ', 'VIQ', 'PIQ']:
    if iq_col in X_pheno_raw.columns: integ_audit.append({'check': f'{iq_col} between 40 and 200', 'status': 'PASS' if X_pheno_raw[iq_col].dropna().between(40, 200).all() else 'FAIL'})

pd.DataFrame(integ_audit).to_csv(RESULT_DIR / 'w1_pheno_integrity_audit.csv', index=False)
assert all(r['status'] == 'PASS' for r in integ_audit), 'Phenotype Integrity checks failed!'

# FIQ Source Consistency Audit
if 'FIQ' in X_pheno_raw.columns and 'FIQ_source' in X_pheno_raw.columns:
    mismatch_1 = X_pheno_raw['FIQ'].notna() & X_pheno_raw['FIQ_source'].isna()
    mismatch_2 = X_pheno_raw['FIQ'].isna() & X_pheno_raw['FIQ_source'].notna()
    iq_audit = [{'check': 'FIQ requires FIQ_source', 'status': 'FAIL' if mismatch_1.any() else 'PASS'}, {'check': 'FIQ_source requires FIQ', 'status': 'FAIL' if mismatch_2.any() else 'PASS'}]
    pd.DataFrame(iq_audit).to_csv(RESULT_DIR / 'w1_iq_integrity_audit.csv', index=False)
    assert (X_pheno_raw['FIQ'].notna() == X_pheno_raw['FIQ_source'].notna()).all(), 'Explicit FIQ consistency assertion failed!'

feat_dict = [{'feature_name': c, 'description': 'Demographic or clinical covariate', 'source': 'ADHD200 phenotypic files', 'encoding': 'Continuous or categorical'} for c in X_pheno_raw.columns]
for d in feat_dict:
    if d['feature_name'] == 'FIQ_source': d.update({'description': 'Source of Full Scale IQ', 'encoding': '0=Full2 IQ, 1=Full4 IQ'})
    elif d['feature_name'] == 'Gender': d.update({'description': 'Gender code', 'encoding': '0=Female, 1=Male'})
    elif d['feature_name'] == 'Handedness':
        h_vals = set(X_pheno_raw['Handedness'].dropna().unique())
        if h_vals.issubset({0, 1, 2, 3, 0.0, 1.0, 2.0, 3.0}): d.update({'description': 'Handedness code', 'encoding': 'Categorical [0, 1, 2, 3]'})
        else: d.update({'description': 'Handedness code', 'encoding': 'Edinburgh [-100, 100]'})
feat_dict.append({'feature_name': 'FC_Edges', 'description': 'Flattened upper-triangle functional connectivity matrices', 'source': 'Generated dynamically from FC_norm'})
feat_dict.append({'feature_name': 'Graph_Metrics', 'description': 'Expanded 23-component topological features over 90th percentile threshold edges', 'source': 'Generated dynamically from absolute FC_norm'})
pd.DataFrame(feat_dict).to_csv(REPORT_DIR / 'w1_feature_dictionary.csv', index=False)


master_cohort columns: ['subject_id', 'site', 'roi_file', 'DX', 'Age', 'Gender', 'Handedness']


In [17]:
X_flat = np.array([fc[np.triu_indices(190, k=1)] for fc in X_fc], dtype=np.float32)
print(f'X_flat shape: {X_flat.shape}')

X_flat shape: (497, 17955)


In [18]:
graph_path = log_file('graph_features', FEATURE_DIR / 'w1_graph_features.parquet')
if (not FORCE_REBUILD_FEATURES) and graph_path.exists(): X_graph = pd.read_parquet(graph_path).values.astype(np.float32)
else:
    def comp(fc):
        a = np.abs(fc.copy())
        np.fill_diagonal(a, 0)
        fv = a[a > 0]
        if len(fv)==0: 
            return {k:0.0 for k in ['degree_mean', 'degree_std', 'degree_p10', 'degree_p25', 'degree_p50', 'degree_p75', 'degree_p90', 'degree_max',
                                    'clustering_mean', 'clustering_std', 'clustering_p10', 'clustering_p25', 'clustering_p50', 'clustering_p75', 'clustering_p90', 'clustering_max',
                                    'density', 'global_efficiency', 'local_efficiency', 'transitivity', 'assortativity', 'n_components', 'largest_component_ratio']}
        G = nx.from_numpy_array((a >= np.percentile(fv, 90)).astype(float))
        degs = [d for _, d in G.degree()]
        clus = list(nx.clustering(G).values())
        try: assort = float(nx.degree_assortativity_coefficient(G))
        except: assort = 0.0
        if np.isnan(assort): assort = 0.0
        comps = list(nx.connected_components(G))
        lc_ratio = len(max(comps, key=len)) / float(len(G)) if comps else 0.0
        return {'degree_mean': float(np.mean(degs)), 'degree_std': float(np.std(degs)), 
                'degree_p10': float(np.percentile(degs, 10)), 'degree_p25': float(np.percentile(degs, 25)), 'degree_p50': float(np.percentile(degs, 50)), 'degree_p75': float(np.percentile(degs, 75)), 'degree_p90': float(np.percentile(degs, 90)), 'degree_max': float(np.max(degs)),
                'clustering_mean': float(np.mean(clus)), 'clustering_std': float(np.std(clus)), 
                'clustering_p10': float(np.percentile(clus, 10)), 'clustering_p25': float(np.percentile(clus, 25)), 'clustering_p50': float(np.percentile(clus, 50)), 'clustering_p75': float(np.percentile(clus, 75)), 'clustering_p90': float(np.percentile(clus, 90)), 'clustering_max': float(np.max(clus)),
                'density': float(nx.density(G)), 'global_efficiency': float(nx.global_efficiency(G)), 'local_efficiency': float(nx.local_efficiency(G)), 'transitivity': float(nx.transitivity(G)), 'assortativity': assort,
                'n_components': float(len(comps)), 'largest_component_ratio': float(lc_ratio)}
    X_graph = pd.DataFrame([comp(X_fc[i]) for i in tqdm(range(len(X_fc)), desc='Graphs')])
    X_graph.to_parquet(graph_path, index=False)
    X_graph = X_graph.values.astype(np.float32)
print(f'X_graph shape: {X_graph.shape}')

Graphs: 100%|██████████| 497/497 [05:18<00:00,  1.56it/s]

X_graph shape: (497, 23)


In [19]:
site_l = LabelEncoder().fit_transform(sites)
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# 1. LOSO ADHD Signal Evaluation
loso_adhd_probs = np.zeros(len(y))
for t_site in np.unique(sites):
    tr, ts = sites != t_site, sites == t_site
    if len(np.unique(y[tr])) < 2: continue
    clf.fit(X_flat[tr], y[tr])
    loso_adhd_probs[ts] = clf.predict_proba(X_flat[ts])[:,1]

a_p_loso = (loso_adhd_probs >= 0.5).astype(int)
a_ba_loso = balanced_accuracy_score(y, a_p_loso)
a_auc_loso = roc_auc_score(y, loso_adhd_probs)
a_f1_loso = f1_score(y, a_p_loso, average='macro')
pd.DataFrame([{'target':'adhd', 'bal_acc':a_ba_loso, 'macro_f1':a_f1_loso, 'auc':a_auc_loso}]).to_csv(RESULT_DIR / 'w1_adhd_signal_loso.csv', index=False)

# 2. Exploratory Site Signal (Not comparable to LOSO Benchmark)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
s_p = cross_val_predict(clf, X_flat, site_l, cv=cv)
s_ba = balanced_accuracy_score(site_l, s_p)
s_f1 = f1_score(site_l, s_p, average='macro')
pd.DataFrame([{'target':'site', 'bal_acc':s_ba, 'macro_f1':s_f1}]).to_csv(RESULT_DIR / 'w1_site_signal_exploratory.csv', index=False)


In [67]:
from joblib import Parallel, delayed
from sklearn.exceptions import ConvergenceWarning

def impute_other(X_tr, X_ts):
    AUDIT_STATE['imputer_train_only'] = True
    tr, ts = X_tr.copy(), X_ts.copy()
    imp = {}
    for c in range(tr.shape[1]):
        if np.isnan(tr[:,c]).any() or np.isnan(ts[:,c]).any():
            med = np.nanmedian(tr[:,c])
            if np.isnan(med): med = 0.0
            tr[np.isnan(tr[:,c]), c] = med
            ts[np.isnan(ts[:,c]), c] = med
            imp[c] = med
    return tr, ts, imp

VERBOSE = False
TREE_N_JOBS = 1
PARAM_N_JOBS = 16

def _fit_one_config(fam_name, t_site, m_name, p, k, make_fn, fold_cache, y_tr, seed):
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)

    p_search = dict(p)
    if m_name == 'SVM':
        search_make_fn = lambda s, **pp: SVC(probability=False, class_weight='balanced', random_state=s, **pp)
    else:
        search_make_fn = make_fn

    f_sc = []
    k_act = None
    for fold in fold_cache:
        Xt_s, Xv_s, k_act = fold['scaled'][k]
        mdl = search_make_fn(s=seed, **p_search)
        mdl.fit(Xt_s, y_tr[fold['i_tr']])
        prob = mdl.predict_proba(Xv_s)[:,1] if hasattr(mdl,'predict_proba') else mdl.decision_function(Xv_s)
        try: f_sc.append(roc_auc_score(y_tr[fold['i_vl']], prob))
        except: f_sc.append(0.5)
    mean_auc = np.mean(f_sc)
    return {'model': m_name, 'params': p, 'param_str': str(p), 'k': k_act, 'auc': mean_auc}


def evaluate_family(fam_name, X_main, X_other, y, sites, models_dict, top_ks=[500, 1000, 2000], seed=42, progress_bar=None):
    all_res = []
    pred_records = []
    fold_features = []
    selection_stability = []
    ensemble_manifests = []
    all_inner_scores = []   # NEW: collect every inner score across all sites
    param_cache = {m_name: list(ParameterGrid(param_grid)) for m_name, (param_grid, _) in models_dict.items()}

    site_list = np.unique(sites)
    t0 = time.time()

    for t_site in site_list:
        tr, ts = sites != t_site, sites == t_site
        y_tr, y_ts = y[tr], y[ts]
        n_subj, n_adhd, n_ctrl = len(y_ts), int((y_ts==1).sum()), int((y_ts==0).sum())

        if len(np.unique(y_ts)) < 2:
            all_res.append({'family': fam_name, 'test_site': t_site, 'model': 'Ensemble', 'degenerate': True, 'n_subjects': n_subj, 'n_adhd': n_adhd, 'n_control': n_ctrl})
            continue

        Xm_tr = X_main[tr] if X_main is not None else None
        Xm_ts = X_main[ts] if X_main is not None else None
        Xo_tr, Xo_ts, imp = impute_other(X_other[tr], X_other[ts]) if X_other is not None else (None, None, None)

        cv = StratifiedKFold(3, shuffle=True, random_state=seed)
        cv_splits = list(cv.split(np.zeros(len(y_tr)), y_tr))
        fold_cache = []
        for i_tr, i_vl in cv_splits:
            cache_item = {'i_tr': i_tr, 'i_vl': i_vl, 'scaled': {}}
            if Xm_tr is not None:
                AUDIT_STATE['selectkbest_train_only'] = True
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    scores, _ = f_classif(Xm_tr[i_tr], y_tr[i_tr])
                scores[np.isnan(scores)] = -np.inf
                ranking = np.argsort(scores)[::-1]
            else: ranking = None

            ks_to_try = top_ks if Xm_tr is not None else [1]
            for k in ks_to_try:
                k_act = min(k, Xm_tr.shape[1]) if Xm_tr is not None else 1
                if Xm_tr is not None:
                    top_k_idx = ranking[:k_act]
                    Xt_sel = Xm_tr[i_tr][:, top_k_idx]
                    Xv_sel = Xm_tr[i_vl][:, top_k_idx]
                else: Xt_sel, Xv_sel = np.empty((len(i_tr), 0)), np.empty((len(i_vl), 0))

                if Xo_tr is not None:
                    Xt_sel = np.hstack([Xt_sel, Xo_tr[i_tr]])
                    Xv_sel = np.hstack([Xv_sel, Xo_tr[i_vl]])

                AUDIT_STATE['scaler_train_only'] = True
                sc = StandardScaler()
                Xt_s = sc.fit_transform(Xt_sel)
                Xv_s = sc.transform(Xv_sel)
                cache_item['scaled'][k] = (Xt_s, Xv_s, k_act)
            fold_cache.append(cache_item)

        full_ranking = None
        if Xm_tr is not None:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                full_scores, _ = f_classif(Xm_tr, y_tr)
            full_scores[np.isnan(full_scores)] = -np.inf
            full_ranking = np.argsort(full_scores)[::-1]

        jobs = []
        for m_name, (_, make_fn) in models_dict.items():
            grid = param_cache[m_name]
            ks_to_try = top_ks if X_main is not None else [1]
            for p in grid:
                for k in ks_to_try:
                    jobs.append((m_name, p, k, make_fn))

        inner_scores = Parallel(n_jobs=PARAM_N_JOBS, backend='loky', verbose=0)(
            delayed(_fit_one_config)(fam_name, t_site, m_name, p, k, make_fn, fold_cache, y_tr, seed)
            for (m_name, p, k, make_fn) in jobs
        )

        # NEW: tag each inner score with site and family, save for full reporting
        for r in inner_scores:
            r['family'] = fam_name
            r['test_site'] = t_site
        all_inner_scores.extend(inner_scores)

        if progress_bar is not None:
            progress_bar.update(len(jobs))

        best_this_site = max(inner_scores, key=lambda r: r['auc'])
        msg = (f'[{fam_name}] {t_site} done  n={n_subj} (ADHD={n_adhd}, ctrl={n_ctrl})  '
               f'{len(jobs)} configs  best={best_this_site["model"]} k={best_this_site["k"]} '
               f'AUC={best_this_site["auc"]:.4f}  elapsed={(time.time()-t0)/60:.1f}min')
        progress_bar.write(msg) if progress_bar else print(msg)

        if VERBOSE:
            for r in inner_scores:
                line = f'[{fam_name}] [{t_site}] [{r["model"]}] k={r["k"]} AUC={r["auc"]:.4f}'
                progress_bar.write(line) if progress_bar else print(line)

        inner_df = pd.DataFrame(inner_scores)
        best_inner = inner_df.loc[inner_df['auc'].idxmax()]
        selection_stability.append({'family': fam_name, 'test_site': t_site, 'selected_model': best_inner['model'], 'selected_params': best_inner['param_str']})

        best_per_model = inner_df.loc[inner_df.groupby('model')['auc'].idxmax()]
        top_3 = best_per_model.nlargest(3, 'auc')

        # ── NEW: refit every model on full train, evaluate on test site ──
        # This is what populates all_res correctly for all models x all sites
        for _, r in best_per_model.iterrows():
            m_name, p, param_str, k, inner_auc = r['model'], r['params'], r['param_str'], r['k'], r['auc']

            if Xm_tr is not None:
                k_act = min(k, Xm_tr.shape[1])
                top_k_idx = full_ranking[:k_act]
                Xtr_s = Xm_tr[:, top_k_idx]
                Xts_s = Xm_ts[:, top_k_idx]
            else:
                Xtr_s, Xts_s = np.empty((len(y_tr), 0)), np.empty((len(y_ts), 0))

            if Xo_tr is not None:
                Xtr_s = np.hstack([Xtr_s, Xo_tr])
                Xts_s = np.hstack([Xts_s, Xo_ts])

            sc = StandardScaler()
            Xtr_s = sc.fit_transform(Xtr_s)
            Xts_s = sc.transform(Xts_s)

            make_fn = models_dict[m_name][1]
            if m_name == 'SVM':
                mdl = SVC(probability=True, class_weight='balanced', random_state=seed, **p)
            else:
                mdl = make_fn(s=seed, **p)

            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                mdl.fit(Xtr_s, y_tr)

            prob = mdl.predict_proba(Xts_s)[:,1] if hasattr(mdl,'predict_proba') else mdl.decision_function(Xts_s)
            ba = balanced_accuracy_score(y_ts, (prob>=0.5).astype(int))
            test_auc = roc_auc_score(y_ts, prob)

            all_res.append({
                'family': fam_name, 'test_site': t_site, 'model': m_name,
                'params': p, 'param_str': param_str, 'k': k,
                'inner_auc': inner_auc, 'auc': test_auc, 'ba': ba,
                'degenerate': False, 'is_ensemble': False,
                'n_subjects': n_subj, 'n_adhd': n_adhd, 'n_control': n_ctrl
            })
            pred_records.append({
                'family': fam_name, 'test_site': t_site, 'model': m_name,
                'is_ensemble': False, 'prob': prob.tolist(), 'y_true': y_ts.tolist()
            })

        # ── Ensemble: top-3 only (unchanged, review said this is fine) ──
        ensemble_probs = np.zeros(len(y_ts))
        weight_sum = 0.0
        ens_manifest_row = []

        for idx, r in top_3.iterrows():
            m_name, p, param_str, k, auc = r['model'], r['params'], r['param_str'], r['k'], r['auc']

            if Xm_tr is not None:
                k_act = min(k, Xm_tr.shape[1])
                top_k_idx = full_ranking[:k_act]
                Xtr_s = Xm_tr[:, top_k_idx]
                Xts_s = Xm_ts[:, top_k_idx]
                if idx == top_3.index[0]: fold_features.append(top_k_idx)
            else:
                Xtr_s, Xts_s = np.empty((len(y_tr), 0)), np.empty((len(y_ts), 0))

            if Xo_tr is not None:
                Xtr_s = np.hstack([Xtr_s, Xo_tr])
                Xts_s = np.hstack([Xts_s, Xo_ts])

            sc = StandardScaler()
            Xtr_s = sc.fit_transform(Xtr_s)
            Xts_s = sc.transform(Xts_s)

            make_fn = models_dict[m_name][1]
            if m_name == 'SVM':
                mdl = SVC(probability=True, class_weight='balanced', random_state=seed, **p)
            else:
                mdl = make_fn(s=seed, **p)

            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                mdl.fit(Xtr_s, y_tr)

            prob = mdl.predict_proba(Xts_s)[:,1] if hasattr(mdl,'predict_proba') else mdl.decision_function(Xts_s)
            w = max(auc - 0.5, 0.001)
            ensemble_probs += prob * w
            weight_sum += w
            ens_manifest_row.append({'member_model': m_name, 'member_params': p, 'member_inner_auc': auc, 'member_weight': w})

        ensemble_probs /= weight_sum
        ba = balanced_accuracy_score(y_ts, (ensemble_probs>=0.5).astype(int))
        all_res.append({
            'family': fam_name, 'test_site': t_site, 'model': 'Ensemble',
            'params': {}, 'param_str': '{}', 'k': None,
            'inner_auc': None, 'auc': roc_auc_score(y_ts, ensemble_probs), 'ba': ba,
            'degenerate': False, 'is_ensemble': True,
            'n_subjects': n_subj, 'n_adhd': n_adhd, 'n_control': n_ctrl
        })
        pred_records.append({
            'family': fam_name, 'test_site': t_site, 'model': 'Ensemble',
            'is_ensemble': True, 'prob': ensemble_probs.tolist(), 'y_true': y_ts.tolist()
        })
        ensemble_manifests.append({'family': fam_name, 'test_site': t_site, 'members': ens_manifest_row})

        with open(RESULT_DIR / 'workflow1_progress.json', 'w') as f:
            json.dump({'family': fam_name, 'site': t_site, 'timestamp': str(datetime.now())}, f, indent=2)

    pd.DataFrame(pred_records).to_parquet(RESULT_DIR / f'w1_predictions_{fam_name}.parquet', index=False)
    # NEW: save full inner scores so hyperparameter analysis has complete data
    pd.DataFrame(all_inner_scores).to_parquet(RESULT_DIR / f'w1_inner_scores_{fam_name}.parquet', index=False)
    return pd.DataFrame(all_res), fold_features, selection_stability, ensemble_manifests

In [69]:
import warnings
warnings.simplefilter("ignore", FutureWarning)
X_pheno_df = pd.read_parquet(FEATURE_DIR / 'w1_pheno_features.parquet')
X_pheno_np = X_pheno_df.values.astype(np.float32)
X_pheno_noiq_df = X_pheno_df.drop(columns=['VIQ', 'PIQ', 'FIQ', 'FIQ_source', 'VIQ_missing', 'PIQ_missing', 'FIQ_missing'], errors='ignore')
assert not any(c in X_pheno_noiq_df.columns for c in ['VIQ', 'PIQ', 'FIQ', 'FIQ_source', 'VIQ_missing', 'PIQ_missing', 'FIQ_missing'])
X_pheno_noiq = X_pheno_noiq_df.values.astype(np.float32)

models_dict = {
    'LR':  ({'C': [0.001, 0.01, 0.1, 1, 10, 100]}, lambda s, **p: LogisticRegression(class_weight='balanced', max_iter=1000, random_state=s, **p)),
    'EN':  ({'l1_ratio': [0.1, 0.5, 0.9], 'C': [0.01, 0.1, 1, 10]}, lambda s, **p: LogisticRegression(penalty='elasticnet', solver='saga', class_weight='balanced', max_iter=3000, tol=1e-3, random_state=s, **p)),
    'SVM': ({'C': [0.01, 0.1, 1, 10], 'gamma': ['scale', 'auto']}, lambda s, **p: SVC(probability=True, class_weight='balanced', random_state=s, **p)),
    'RF':  ({'n_estimators': [300], 'max_depth': [5, 10]}, lambda s, **p: RandomForestClassifier(n_jobs=TREE_N_JOBS, class_weight='balanced', random_state=s, **p)),
    'ET':  ({'n_estimators': [300], 'max_depth': [5, 10]}, lambda s, **p: ExtraTreesClassifier(n_jobs=TREE_N_JOBS, class_weight='balanced', random_state=s, **p)),
    'XGB': ({'max_depth': [3, 5], 'n_estimators': [200], 'scale_pos_weight': [1, 2.5]}, lambda s, **p: XGBClassifier(n_jobs=TREE_N_JOBS, use_label_encoder=False, eval_metric='logloss', verbosity=0, random_state=s, **p))
}
try:
    models_dict['LGBM'] = ({'num_leaves': [31, 63], 'n_estimators': [200], 'scale_pos_weight': [1, 2.5]}, lambda s, **p: LGBMClassifier(n_jobs=TREE_N_JOBS, verbosity=-1, random_state=s, **p))
except: pass

ff_configs = {
    'FC_only':          (X_flat,  None,       [100, 200, 500, 1000, 2000, 3000, 5000]),
    'Graph_only':       (X_graph, None,       [X_graph.shape[1]]),
    'Phenotype_only':   (None,    X_pheno_np, [1]),
    'Phenotype_NoIQ':   (None,    X_pheno_noiq, [1]),
    'FC_Phenotype':     (X_flat,  X_pheno_np, [100, 200, 500, 1000, 2000, 3000, 5000]),
    'Graph_Phenotype':  (X_graph, X_pheno_np, [X_graph.shape[1]])
}

results = []
stability_features = {}
sel_stabs = []
global_ens_manifest = []
done_fams = 0
total_fams = len(ff_configs)

for f_name, (Xm, Xo, ks) in ff_configs.items():
    fam_res_path   = RESULT_DIR / f'w1_fam_res_{f_name}.parquet'
    fam_feats_path = RESULT_DIR / f'w1_fam_feats_{f_name}.json'
    fam_stab_path  = RESULT_DIR / f'w1_fam_stab_{f_name}.json'
    fam_ens_path   = RESULT_DIR / f'w1_fam_ens_{f_name}.json'

    if USE_CHECKPOINTS and fam_res_path.exists():
        print(f'[{done_fams+1}/{total_fams}] Skipping {f_name} (loaded from checkpoint)')
        results.append(pd.read_parquet(fam_res_path))
        if fam_feats_path.exists():
            with open(fam_feats_path, 'r') as f: stability_features[f_name] = [np.array(x) for x in json.load(f)]
        if fam_stab_path.exists():
            with open(fam_stab_path, 'r') as f: sel_stabs.extend(json.load(f))
        if fam_ens_path.exists():
            with open(fam_ens_path, 'r') as f: global_ens_manifest.extend(json.load(f))
        done_fams += 1
        continue

    n_sites    = len(np.unique(sites))
    ks_count   = len(ks) if Xm is not None else 1
    total_fits = sum(len(list(ParameterGrid(g))) for g, _ in models_dict.values()) * ks_count * n_sites
    print(f'[{done_fams+1}/{total_fams}] Running {f_name}  '
          f'(models={list(models_dict.keys())}, k={ks}, ~{total_fits} total configs)')

    with tqdm(total=total_fits, desc=f_name, unit='cfg') as pbar:
        df, feats, sel_stab, ens_man = evaluate_family(
            f_name, Xm, Xo, y, sites, models_dict, ks, seed=42, progress_bar=pbar
        )

    df.to_parquet(fam_res_path, index=False)   # checkpoint immediately
    results.append(df)
    sel_stabs.extend(sel_stab)
    global_ens_manifest.extend(ens_man)
    if feats: stability_features[f_name] = feats

    done_fams += 1
    elapsed = time.time() - START_TIME
    print(f'[{done_fams}/{total_fams}] {f_name} complete.  Elapsed={elapsed/60:.1f} min  ETA={eta(done_fams, total_fams)}')
    if feats:
        with open(fam_feats_path, 'w') as f: json.dump([x.tolist() for x in feats], f)
    with open(fam_stab_path, 'w') as f: json.dump(sel_stab, f)
    with open(fam_ens_path,  'w') as f: json.dump(ens_man, f)

with open(RESULT_DIR / 'w1_ensemble_manifest.json', 'w') as f:
    json.dump(global_ens_manifest, f, indent=2)

all_res = pd.concat(results)
# exclude ensemble rows from per-model summary; keep only individual model rows
v = all_res[~all_res['degenerate'] & ~all_res['is_ensemble']].copy()

def mean_ci(data, conf=0.95):
    n = len(data)
    se = np.std(data, ddof=1) / np.sqrt(n)
    h = se * st.t.ppf((1 + conf) / 2., n-1)
    return data.mean() - h, data.mean() + h

# ── Summary: now every model has 7 rows (one per LOSO site) ──
sum_data = []
for (fam, mod), grp in v.groupby(['family', 'model']):
    auc_v = grp['auc'].values
    ba_v  = grp['ba'].values
    n     = len(auc_v)
    ci_l, ci_u = mean_ci(auc_v) if n > 1 else (auc_v[0], auc_v[0])
    sum_data.append({
        'family': fam, 'model': mod,
        'mean_auc': auc_v.mean(), 'std_auc': auc_v.std(ddof=1) if n > 1 else 0.0,
        'ci_lower': ci_l, 'ci_upper': ci_u,
        'mean_ba': ba_v.mean(), 'std_ba': ba_v.std(ddof=1) if n > 1 else 0.0,
        'n_folds': n,
        'ci_method': 'Student t distribution over LOSO sites',
        'n_sites': int(len(np.unique(sites))),
        'degrees_of_freedom': int(n - 1)
    })

summary = pd.DataFrame(sum_data)
summary.to_csv(RESULT_DIR / 'w1_model_summary.csv', index=False)

# ── Family winners ──
idx = summary.groupby('family')['mean_auc'].idxmax()
family_winners = summary.loc[idx].copy()
family_winners.to_csv(RESULT_DIR / 'w1_family_winners.csv', index=False)
print('\nFamily Winners (mean ± std AUC):')
for _, r in family_winners.iterrows():
    print(f"  {r['family']} ({r['model']}): {r['mean_auc']:.3f} ± {r['std_auc']:.3f}  [n_folds={r['n_folds']}]")

overall_winner = summary.loc[summary['mean_auc'].idxmax()]
print(f"\nOverall Winner: {overall_winner['family']} ({overall_winner['model']}) "
      f"AUC {overall_winner['mean_auc']:.3f} ± {overall_winner['std_auc']:.3f}  [n_folds={overall_winner['n_folds']}]")

# ── Ablation ──
def get_m(f): return summary.loc[summary['family'] == f, 'mean_auc'].max()
abl_res = [
    {'comparison': 'Phenotype vs Phenotype_NoIQ', 'base': get_m('Phenotype_only'),  'ablated': get_m('Phenotype_NoIQ'), 'delta': get_m('Phenotype_only')  - get_m('Phenotype_NoIQ')},
    {'comparison': 'FC+Pheno vs FC',              'base': get_m('FC_Phenotype'),     'ablated': get_m('FC_only'),        'delta': get_m('FC_Phenotype')     - get_m('FC_only')},
    {'comparison': 'Graph+Pheno vs Graph',         'base': get_m('Graph_Phenotype'),  'ablated': get_m('Graph_only'),     'delta': get_m('Graph_Phenotype')  - get_m('Graph_only')},
]
pd.DataFrame(abl_res).to_csv(RESULT_DIR / 'w1_ablation_deltas.csv', index=False)
summary[summary['family'].isin(['Phenotype_only', 'Phenotype_NoIQ'])].to_csv(RESULT_DIR / 'w1_iq_ablation.csv', index=False)

# ── Site generalisation: now has all 7 sites ──
ow_df    = v[(v['family'] == overall_winner['family']) & (v['model'] == overall_winner['model'])]
site_gen = ow_df[['test_site', 'n_subjects', 'n_adhd', 'n_control', 'auc', 'ba']].copy()
site_gen.to_csv(RESULT_DIR / 'w1_site_generalization.csv', index=False)

# ── Hyperparameter summary: now uses full inner_scores across all sites ──
inner_frames = []
for f_name in ff_configs:
    inner_path = RESULT_DIR / f'w1_inner_scores_{f_name}.parquet'
    if inner_path.exists():
        inner_frames.append(pd.read_parquet(inner_path))
if inner_frames:
    all_inner = pd.concat(inner_frames)
    hp_sum = (all_inner
              .groupby(['family', 'test_site', 'model', 'param_str'])['auc']
              .mean()
              .reset_index()
              .groupby(['family', 'model', 'param_str'])
              .agg(mean_inner_auc=('auc', 'mean'), std_inner_auc=('auc', 'std'), fold_count=('auc', 'count'))
              .reset_index())
    hp_sum.to_csv(RESULT_DIR / 'hyperparameter_summary.csv', index=False)

# ── Oracle stability ──
oracle_stab = []
for (fam, ts), grp in v.groupby(['family', 'test_site']):
    best_for_site = grp.loc[grp['auc'].idxmax()]
    oracle_stab.append({
        'family': fam, 'test_site': ts,
        'winning_model': best_for_site['model'],
        'winning_params': best_for_site['param_str'],
        'auc': best_for_site['auc'], 'ba': best_for_site['ba']
    })
pd.DataFrame(oracle_stab).to_csv(RESULT_DIR / 'w1_oracle_model_stability.csv', index=False)

# ── Selection stability ──
sel_stab_df = pd.DataFrame(sel_stabs)
sel_stab_df.to_csv(RESULT_DIR / 'w1_selection_stability.csv', index=False)
msel_freq = sel_stab_df['selected_model'].value_counts().reset_index()
msel_freq.columns = ['model', 'times_selected']
msel_freq.to_csv(RESULT_DIR / 'w1_selection_frequency.csv', index=False)

[1/6] Running FC_only  (models=['LR', 'EN', 'SVM', 'RF', 'ET', 'XGB', 'LGBM'], k=[100, 200, 500, 1000, 2000, 3000, 5000], ~1862 total configs)


FC_only:  14%|█▍        | 266/1862 [02:22<14:12,  1.87cfg/s]

[FC_only] KKI done  n=83 (ADHD=22, ctrl=61)  266 configs  best=ET k=5000 AUC=0.6480  elapsed=2.4min


FC_only:  29%|██▊       | 532/1862 [06:00<15:34,  1.42cfg/s]

[FC_only] NYU done  n=93 (ADHD=54, ctrl=39)  266 configs  best=LR k=500 AUC=0.6659  elapsed=6.0min


...

[Graph_Phenotype] Peking_2 done  n=67 (ADHD=35, ctrl=32)  38 configs  best=ET k=23 AUC=0.6673  elapsed=1.2min


Graph_Phenotype: 100%|██████████| 266/266 [01:19<00:00,  3.20cfg/s]

[Graph_Phenotype] Peking_3 done  n=42 (ADHD=19, ctrl=23)  38 configs  best=LR k=23 AUC=0.6582  elapsed=1.3min


Graph_Phenotype: 100%|██████████| 266/266 [01:26<00:00,  3.07cfg/s]

[6/6] Graph_Phenotype complete.  Elapsed=169.7 min  ETA=0h 0m

Family Winners (mean ± std AUC):
  FC_Phenotype (ET): 0.633 ± 0.093  [n_folds=7]
  FC_only (LR): 0.614 ± 0.091  [n_folds=7]
  Graph_Phenotype (SVM): 0.649 ± 0.100  [n_folds=7]
  Graph_only (RF): 0.569 ± 0.054  [n_folds=7]
  Phenotype_NoIQ (EN): 0.593 ± 0.103  [n_folds=7]
  Phenotype_only (EN): 0.593 ± 0.103  [n_folds=7]

Overall Winner: Graph_Phenotype (SVM) AUC 0.649 ± 0.100  [n_folds=7]


In [70]:
summary[summary['family'] == 'FC_Phenotype'].sort_values('mean_auc', ascending=False)

,family,model,mean_auc,std_auc,ci_lower,ci_upper,mean_ba,std_ba,n_folds,ci_method,n_sites,degrees_of_freedom
1,FC_Phenotype,ET,0.632847,0.093154,0.546693,0.719000,0.602463,0.065664,7,Student t distribution over LOSO sites,7,6
3,FC_Phenotype,LR,0.606806,0.085532,0.527701,0.685910,0.592084,0.078420,7,Student t distribution over LOSO sites,7,6
4,FC_Phenotype,RF,0.603547,0.092103,0.518366,0.688728,0.541206,0.040403,7,Student t distribution over LOSO sites,7,6
0,FC_Phenotype,EN,0.591696,0.088925,0.509454,0.673937,0.575199,0.050564,7,Student t distribution over LOSO sites,7,6
2,FC_Phenotype,LGBM,0.574424,0.094277,0.487232,0.661616,0.558108,0.055437,7,Student t distribution over LOSO sites,7,6
6,FC_Phenotype,XGB,0.540835,0.100185,0.448179,0.633490,0.543357,0.075334,7,Student t distribution over LOSO sites,7,6
5,FC_Phenotype,SVM,0.455383,0.141289,0.324713,0.586053,0.530492,0.038388,7,Student t distribution over LOSO sites,7,6


In [71]:
v.groupby(['family','model']).size().sort_values(ascending=False)

family           model
FC_Phenotype     EN       7
                 ET       7
                 LGBM     7
                 LR       7
                 RF       7
                 SVM      7
                 XGB      7
FC_only          EN       7
                 ET       7
                 LGBM     7
                 LR       7
                 RF       7
                 SVM      7
                 XGB      7
Graph_Phenotype  EN       7
                 ET       7
                 LGBM     7
                 LR       7
                 RF       7
                 SVM      7
                 XGB      7
Graph_only       EN       7
                 ET       7
                 LGBM     7
                 LR       7
                 RF       7
                 SVM      7
                 XGB      7
Phenotype_NoIQ   EN       7
                 ET       7
                 LGBM     7
                 LR       7
                 RF       7
                 SVM      7
                 XGB     

In [72]:
v[v['family']=='FC_Phenotype'][['test_site','model','auc']].sort_values(['test_site','auc'], ascending=[True,False])

,test_site,model,auc
1,KKI,ET,0.485842
3,KKI,LR,0.463487
2,KKI,LGBM,0.461252
5,KKI,SVM,0.447839
0,KKI,EN,0.446349
4,KKI,RF,0.444858
6,KKI,XGB,0.421013
9,NYU,ET,0.582621
11,NYU,LR,0.547483
13,NYU,SVM,0.540361


In [73]:
stab_records = []
for f_name, feats in stability_features.items():
    if not feats: continue
    counts = pd.Series(np.concatenate(feats)).value_counts()
    for feat_idx, count in counts.items():
        stab_records.append({'family': f_name, 'feature_index': feat_idx, 'selection_count': count, 'total_folds': len(feats)})
pd.DataFrame(stab_records).to_csv(RESULT_DIR / 'w1_feature_stability.csv', index=False)


In [74]:
def load_model_with_warning(filepath):
    print('WARNING: This model was retrained on the full cohort after evaluation.')
    print('Reported LOSO metrics were obtained from fold-specific models.')
    print('This artifact was never evaluated as a held-out model.')
    return joblib.load(filepath)

def refit_and_save(f_name, m_name, summary_row, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    Xm, Xo, _ = ff_configs[f_name]
    
    b_hp_row = best_hp[(best_hp['family']==f_name) & (best_hp['model']==m_name)]
    if not b_hp_row.empty:
        b_p_str = b_hp_row.iloc[0]['param_str']
        df_slice = v[(v['family']==f_name) & (v['model']==m_name) & (v['param_str']==b_p_str)]
        b_p = df_slice['params'].iloc[0] if not df_slice.empty else {}
        b_k = int(df_slice['k'].iloc[0]) if not df_slice.empty and pd.notnull(df_slice['k'].iloc[0]) else None
    else:
        b_p, b_k = {}, None
    
    sel, imp = None, {}
    if Xm is not None and b_k is not None:
        sel = SelectKBest(f_classif, k=b_k)
        Xm_s = sel.fit_transform(Xm, y)
    else: Xm_s = np.empty((len(y), 0))
    
    if Xo is not None:
        Xo_s = Xo.copy()
        for c in range(Xo_s.shape[1]):
            med = np.nanmedian(Xo_s[:,c])
            if np.isnan(med): med = 0.0
            Xo_s[np.isnan(Xo_s[:,c]), c] = med
            imp[c] = med
        X_full = np.hstack([Xm_s, Xo_s])
    else: X_full = Xm_s
    
    sc = StandardScaler()
    X_full = sc.fit_transform(X_full)
    
    if m_name == 'Ensemble':
        mdl = None
        ens_data = [m for m in global_ens_manifest if m['family'] == f_name]
        with open(out_dir / 'ensemble_manifest.json', 'w') as f: json.dump(ens_data, f, indent=2)
    else:
        mdl = models_dict[m_name][1](s=42, **b_p)
        mdl.fit(X_full, y)
    
    pipeline = {'imputer': imp, 'scaler': sc, 'selector': sel, 'classifier': mdl, 'params': b_p, 'k': b_k}
    joblib.dump(pipeline, out_dir / 'best_model.joblib')
    
    metadata = {
        'feature_family': f_name,
        'model': m_name,
        'date': datetime.now().isoformat(),
        'retrained_on_full_dataset': True,
        'warning': 'Final model retrained on full cohort. Not the evaluation model.',
        'best_hyperparameter_source': 'Mean LOSO AUC',
        'mean_auc': summary_row['mean_auc'],
        'std_auc': summary_row['std_auc'],
        'mean_ba': summary_row['mean_ba'],
        'std_ba': summary_row['std_ba'],
        'n_folds': int(summary_row['n_folds']),
        'ci_method': 'Student t distribution over LOSO sites',
        'n_sites': int(len(np.unique(sites))),
        'degrees_of_freedom': int(len(np.unique(sites)) - 1)
    }
    with open(out_dir / 'metadata.json', 'w') as f: 
        json.dump(metadata, f, indent=2)
    
    with open(out_dir / 'model_card.txt', 'w') as f:
        f.write("WARNING\n\n")
        f.write("This model was retrained on the full cohort after evaluation.\n")
        f.write("Reported LOSO metrics were obtained from fold-specific models.\n")
        f.write("The saved artifact itself was not evaluated as a held-out model.\n")
        f.write("Do not interpret metadata AUC as the performance of this artifact.\n")
    
    if sel is not None:
        pd.DataFrame({'feature_idx': sel.get_support(indices=True)}).to_csv(out_dir / 'selected_features.csv', index=False)

for _, w in family_winners.iterrows():
    if w['model'] != 'Ensemble': refit_and_save(w['family'], w['model'], w, MODEL_DIR / w['family'])

if overall_winner['model'] != 'Ensemble':
    refit_and_save(overall_winner['family'], overall_winner['model'], overall_winner, MODEL_DIR / 'overall_winner')


In [75]:
from pathlib import Path
from sklearn.calibration import calibration_curve
pred_frames = []
for fam in ff_configs.keys():
    pred_file = RESULT_DIR / f"w1_predictions_{fam}.parquet"
    if pred_file.exists():
        pred_frames.append(pd.read_parquet(pred_file))
if len(pred_frames) == 0:
    raise FileNotFoundError(
        "No prediction parquet files found. "
    )
pred_df = pd.concat(pred_frames, ignore_index=True)
plt.figure(figsize=(10, 6))
cal_records = []
for _, w in family_winners.iterrows():
    f_name = w["family"]
    m_name = w["model"]
    df = pred_df[
        (pred_df["family"] == f_name) &
        (pred_df["model"] == m_name)
    ]
    if len(df) == 0:
        print(f"Skipping {f_name}/{m_name}: no predictions found")
        continue
    y_true_all = []
    y_prob_all = []
    for _, row in df.iterrows():
        y_true_all.extend(row["y_true"])
        y_prob_all.extend(row["prob"])
    y_true_all = np.array(y_true_all)
    y_prob_all = np.array(y_prob_all)
    frac_pos, mean_pred = calibration_curve(
        y_true_all,
        y_prob_all,
        n_bins=10,
        strategy="uniform"
    )
    plt.plot(
        mean_pred,
        frac_pos,
        marker="o",
        label=f"{f_name} ({m_name})"
    )
    for fp, mp in zip(frac_pos, mean_pred):
        cal_records.append({
            "family": f_name,
            "model": m_name,
            "prob_pred": mp,
            "prob_true": fp
        })
plt.plot(
    [0, 1],
    [0, 1],
    "k--",
    label="Perfectly calibrated"
)
plt.xlabel("Predicted Probability")
plt.ylabel("Observed Frequency")
plt.title("Calibration Curves of Family Winners")
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "w1_calibration.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()
pd.DataFrame(cal_records).to_csv(
    RESULT_DIR / "w1_calibration.csv",
    index=False
)
print("Calibration analysis complete.")

Calibration analysis complete.


In [76]:
pd.DataFrame(FILE_MANIFEST).to_csv(RESULT_DIR / 'loaded_file_manifest.csv', index=False)
prov = {'timestamp': datetime.now().isoformat(), 'subjects': len(y), 'sites': len(np.unique(sites)), 'file_paths': {f['logical_name']: f['absolute_path'] for f in FILE_MANIFEST}, 'dtypes': X_pheno_df.dtypes.astype(str).to_dict()}
with open(RESULT_DIR / 'w1_data_provenance.json', 'w') as f: json.dump(prov, f, indent=2)

audit_status = 'PASS'
final_audit = []
try:
    for f in FILE_MANIFEST:
        assert '_archive' not in Path(f['absolute_path']).parts, f'Archive file detected: {f["absolute_path"]}'
        assert 'cpac' not in [p.lower() for p in Path(f['absolute_path']).parts], f'CPAC file detected: {f["absolute_path"]}'
    final_audit.append({'check': 'No archive files used', 'status': 'PASS'})
    final_audit.append({'check': 'No CPAC files used', 'status': 'PASS'})

    bad_cols = {'y', 'dx', 'adhd', 'adhd_binary', 'target', 'label', 'diagnosis', 'class', 'group', 'outcome'}
    for c in X_pheno_df.columns:
        assert c.lower() not in bad_cols, f'Target variable {c} leaked into phenotype features'
    final_audit.append({'check': 'No target leakage in phenotype features', 'status': 'PASS'})
    final_audit.append({'check': 'All phenotype dtypes numeric', 'status': 'PASS' if all(np.issubdtype(dt, np.number) for dt in X_pheno_df.dtypes) else 'FAIL'})

    final_audit.append({'check': 'Scaling executed inside fold-wise pipeline', 'status': 'PASS' if AUDIT_STATE.get('scaler_train_only', False) else 'FAIL'})
    final_audit.append({'check': 'Feature selection executed inside fold-wise pipeline', 'status': 'PASS' if AUDIT_STATE.get('selectkbest_train_only', False) else 'FAIL'})
    final_audit.append({'check': 'Imputation executed inside fold-wise pipeline', 'status': 'PASS' if AUDIT_STATE.get('imputer_train_only', False) else 'FAIL'})

    allowed_roots = [
        DATA_ROOT.resolve(),
        RUN_ROOT.resolve()
    ]
    paths_ok = all(
        any(
            Path(f['absolute_path']).resolve().is_relative_to(r)
            for r in allowed_roots
        )
        for f in FILE_MANIFEST
    )
    final_audit.append({'check': 'All paths canonical', 'status': 'PASS' if paths_ok else 'FAIL'})
    final_audit.append({'check': 'All files exist', 'status': 'PASS' if all(f['exists'] for f in FILE_MANIFEST) else 'FAIL'})

except Exception as e:
    audit_status = f'FAIL: {e}'
    final_audit.append({'check': 'Exception Hit', 'status': audit_status})
    pd.DataFrame(final_audit).to_csv(RESULT_DIR / 'w1_final_audit.csv', index=False)
    raise
else:
    pd.DataFrame(final_audit).to_csv(RESULT_DIR / 'w1_final_audit.csv', index=False)

In [77]:
reg_path = REPORT_DIR / 'experiment_registry.csv'
ts = datetime.now().strftime('%Y-%m-%d %H:%M')
new_entries = [{'experiment_id': f'w1_{w["family"]}_{w["model"]}', 'date': ts, 'model': w['model'], 'status': 'COMPLETED'} for _, w in family_winners.iterrows()]
df = pd.DataFrame(new_entries)
if reg_path.exists(): pd.concat([pd.read_csv(reg_path), df]).to_csv(reg_path, index=False)
else: df.to_csv(reg_path, index=False)
print('Registry updated.')

Registry updated.


In [78]:
from pathlib import Path

RUN_ROOT = Path("/mnt/ADHD200_WORKFLOW1_RUN")

for p in sorted(RUN_ROOT.rglob("*")):
    print(p)

/mnt/ADHD200_WORKFLOW1_RUN/04_features
/mnt/ADHD200_WORKFLOW1_RUN/04_features/w1_graph_features.parquet
/mnt/ADHD200_WORKFLOW1_RUN/04_features/w1_pheno_features.parquet
/mnt/ADHD200_WORKFLOW1_RUN/05_models
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_Phenotype
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_Phenotype/best_model.joblib
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_Phenotype/metadata.json
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_Phenotype/model_card.txt
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_Phenotype/selected_features.csv
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_only
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_only/best_model.joblib
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_only/metadata.json
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_only/model_card.txt
/mnt/ADHD200_WORKFLOW1_RUN/05_models/classical_ml/FC_only/selected_features.csv
/

In [61]:
RESULT_DIR = Path("/mnt/ADHD200_WORKFLOW1_RUN/06_results/workflow1")

In [79]:
pd.read_csv(RESULT_DIR / "w1_final_audit.csv")

,check,status
0,No archive files used,PASS
1,No CPAC files used,PASS
2,No target leakage in phenotype features,PASS
3,All phenotype dtypes numeric,PASS
4,Scaling executed inside fold-wise pipeline,PASS
5,Feature selection executed inside fold-wise pi...,PASS
6,Imputation executed inside fold-wise pipeline,PASS
7,All paths canonical,PASS
8,All files exist,PASS


In [80]:
pd.read_csv(RESULT_DIR / "w1_selection_frequency.csv")

,model,times_selected
0,ET,14
1,SVM,13
2,LR,6
3,EN,6
4,XGB,2
5,RF,1


In [81]:
pd.read_csv(
    RESULT_DIR / "w1_ablation_deltas.csv"
)

,comparison,base,ablated,delta
0,Phenotype vs Phenotype_NoIQ,0.593487,0.593487,0.000000
1,FC+Pheno vs FC,0.632847,0.614087,0.018759
2,Graph+Pheno vs Graph,0.649309,0.569260,0.080049


In [82]:
pd.read_csv(
    RESULT_DIR / "w1_site_generalization.csv"
)

,test_site,n_subjects,n_adhd,n_control,auc,ba
0,KKI,83,22,61,0.489940,0.543219
1,NYU,93,54,39,0.641026,0.604701
2,NeuroIMAGE,48,25,23,0.700870,0.556522
3,OHSU,79,37,42,0.680824,0.620978
4,Peking_1,85,24,61,0.797814,0.717896
5,Peking_2,67,35,32,0.676339,0.616071
6,Peking_3,42,19,23,0.558352,0.459954


In [83]:
import pandas as pd

hp = pd.read_csv(
    "/mnt/ADHD200_WORKFLOW1_RUN/06_results/workflow1/hyperparameter_summary.csv"
)

print(hp.shape)
print(hp.head())
print(hp.columns.tolist())

print(
    hp.groupby(["family","model"])
      .size()
      .sort_values(ascending=False)
)

(228, 6)
         family model                     param_str  mean_inner_auc  \
0  FC_Phenotype    EN  {'C': 0.01, 'l1_ratio': 0.1}        0.629777   
1  FC_Phenotype    EN  {'C': 0.01, 'l1_ratio': 0.5}        0.503834   
2  FC_Phenotype    EN  {'C': 0.01, 'l1_ratio': 0.9}        0.500000   
3  FC_Phenotype    EN   {'C': 0.1, 'l1_ratio': 0.1}        0.607798   
4  FC_Phenotype    EN   {'C': 0.1, 'l1_ratio': 0.5}        0.617103   

   std_inner_auc  fold_count  
0       0.023811           7  
1       0.010144           7  
2       0.000000           7  
3       0.028510           7  
4       0.031499           7  
['family', 'model', 'param_str', 'mean_inner_auc', 'std_inner_auc', 'fold_count']
family           model
FC_Phenotype     EN       12
FC_only          EN       12
Phenotype_NoIQ   EN       12
Phenotype_only   EN       12
Graph_only       EN       12
Graph_Phenotype  EN       12
FC_Phenotype     SVM       8
Graph_Phenotype  SVM       8
Phenotype_NoIQ   SVM       8
Graph_only  

In [85]:
pd.set_option("display.max_rows",200)

summary.sort_values(
    ["family","mean_auc"],
    ascending=[True,False]
)

,family,model,mean_auc,std_auc,ci_lower,ci_upper,mean_ba,std_ba,n_folds,ci_method,n_sites,degrees_of_freedom
1,FC_Phenotype,ET,0.632847,0.093154,0.546693,0.719000,0.602463,0.065664,7,Student t distribution over LOSO sites,7,6
3,FC_Phenotype,LR,0.606806,0.085532,0.527701,0.685910,0.592084,0.078420,7,Student t distribution over LOSO sites,7,6
4,FC_Phenotype,RF,0.603547,0.092103,0.518366,0.688728,0.541206,0.040403,7,Student t distribution over LOSO sites,7,6
0,FC_Phenotype,EN,0.591696,0.088925,0.509454,0.673937,0.575199,0.050564,7,Student t distribution over LOSO sites,7,6
2,FC_Phenotype,LGBM,0.574424,0.094277,0.487232,0.661616,0.558108,0.055437,7,Student t distribution over LOSO sites,7,6
6,FC_Phenotype,XGB,0.540835,0.100185,0.448179,0.633490,0.543357,0.075334,7,Student t distribution over LOSO sites,7,6
5,FC_Phenotype,SVM,0.455383,0.141289,0.324713,0.586053,0.530492,0.038388,7,Student t distribution over LOSO sites,7,6
10,FC_only,LR,0.614087,0.090635,0.530264,0.697910,0.590002,0.076300,7,Student t distribution over LOSO sites,7,6
8,FC_only,ET,0.613833,0.081948,0.538044,0.689622,0.583292,0.069551,7,Student t distribution over LOSO sites,7,6
11,FC_only,RF,0.602764,0.081421,0.527463,0.678066,0.572613,0.078351,7,Student t distribution over LOSO sites,7,6


In [86]:
pd.read_parquet("/mnt/ADHD200_WORKFLOW1_RUN/04_features/w1_graph_features.parquet").head()

,degree_mean,degree_std,degree_p10,degree_p25,degree_p50,degree_p75,degree_p90,degree_max,clustering_mean,clustering_std,...,clustering_p75,clustering_p90,clustering_max,density,global_efficiency,local_efficiency,transitivity,assortativity,n_components,largest_component_ratio
0,18.905263,9.601068,7.0,12.0,17.0,24.75,32.0,42.0,0.332248,0.128130,...,0.385440,0.468052,1.000000,0.100028,0.493590,0.566003,0.332885,0.265241,1.0,1.000000
1,18.905263,8.766652,8.0,12.0,18.0,24.00,31.1,46.0,0.343602,0.107359,...,0.399822,0.468190,1.000000,0.100028,0.492319,0.592978,0.350490,0.288318,1.0,1.000000
2,18.905263,10.013227,7.0,12.0,17.0,26.75,32.0,46.0,0.450478,0.149594,...,0.547562,0.627514,0.888889,0.100028,0.460615,0.666503,0.461555,0.379518,2.0,0.994737
3,18.905263,13.332251,4.9,8.0,15.0,27.75,39.1,59.0,0.388879,0.177820,...,0.488570,0.571429,1.000000,0.100028,0.465978,0.573627,0.437729,0.343309,2.0,0.994737
4,18.905263,8.787040,8.0,12.0,18.0,24.00,32.0,43.0,0.333981,0.103494,...,0.390696,0.455245,0.714286,0.100028,0.491784,0.578563,0.336439,0.236205,1.0,1.000000


In [1]:
import os
import psutil

p = psutil.Process(os.getpid())

print("PID:", p.pid)
print("Process name:", p.name())
print("Status:", p.status())
print("Memory (GB):", round(p.memory_info().rss / 1024**3, 2))

PID: 2266258
Process name: python3
Status: running
Memory (GB): 0.07


In [4]:
import pandas as pd
import json
from pathlib import Path

RUN_ROOT = Path('/mnt/ADHD200_WORKFLOW1_RUN')

print("="*70)
print("WORKFLOW 1 RESULTS DASHBOARD")
print("="*70)

# 1. Model summary
df = pd.read_csv(RUN_ROOT / "06_results/workflow1/w1_model_summary.csv")
print("\n📊 Model Summary (best per family?):")
print(df.to_string(index=False))

# 2. Family winners
df_w = pd.read_csv(RUN_ROOT / "06_results/workflow1/w1_family_winners.csv")
print("\n🏆 Family Winners:")
print(df_w.to_string(index=False))

# 3. Full ranking (if exists)
rank_path = RUN_ROOT / "06_results/06_results/workflow1/w1_full_model_ranking.csv"
if rank_path.exists():
    df_r = pd.read_csv(rank_path)
    print("\n📈 Full Model Ranking (top 5):")
    print(df_r.head(5).to_string(index=False))

# 4. Hyperparameters
df_hp = pd.read_csv(RUN_ROOT / "06_results/workflow1/hyperparameter_summary.csv")
print("\n⚙️ Hyperparameter Summary (first 5 rows):")
print(df_hp.head(5).to_string(index=False))

# 5. Ensemble JSON example
ens_file = RUN_ROOT / "06_results/workflow1/w1_fam_ens_FC_Phenotype.json"
if ens_file.exists():
    with open(ens_file) as f:
        ens = json.load(f)
    print("\n🧩 Ensemble (FC+Phenotype):")
    print(json.dumps(ens, indent=2)[:500] + "...")

WORKFLOW 1 RESULTS DASHBOARD

📊 Model Summary (best per family?):
         family model  mean_auc  std_auc  ci_lower  ci_upper  mean_ba   std_ba  n_folds                              ci_method  n_sites  degrees_of_freedom
   FC_Phenotype    EN  0.591696 0.088925  0.509454  0.673937 0.575199 0.050564        7 Student t distribution over LOSO sites        7                   6
   FC_Phenotype    ET  0.632847 0.093154  0.546693  0.719000 0.602463 0.065664        7 Student t distribution over LOSO sites        7                   6
   FC_Phenotype  LGBM  0.574424 0.094277  0.487232  0.661616 0.558108 0.055437        7 Student t distribution over LOSO sites        7                   6
   FC_Phenotype    LR  0.606806 0.085532  0.527701  0.685910 0.592084 0.078420        7 Student t distribution over LOSO sites        7                   6
   FC_Phenotype    RF  0.603547 0.092103  0.518366  0.688728 0.541206 0.040403        7 Student t distribution over LOSO sites        7                   

In [8]:
import pandas as pd
import json
from pathlib import Path

W2_ROOT = Path('/mnt/ADHD200_WORKFLOW2_RUN')

print("=" * 80)
print("WORKFLOW 2 – MAIN RESULTS (FIXED)")
print("=" * 80)

# ---------- 1. GNN Summary (best per architecture) ----------
summary_file = W2_ROOT / "06_results/workflow2b/w2b_summary_with_ci.csv"
if summary_file.exists():
    df = pd.read_csv(summary_file)
    print("\n📊 GNN SUMMARY (BEST PER ARCHITECTURE):")
    print(df.to_string(index=False))
else:
    print("❌ w2b_summary_with_ci.csv not found")

# ---------- 2. Classical vs GNN comparison ----------
compare_file = W2_ROOT / "06_results/workflow2b/w2b_classical_vs_gnn.csv"
if compare_file.exists():
    df_comp = pd.read_csv(compare_file)
    print("\n📈 CLASSICAL vs GNN (AUC / Balanced Acc):")
    # Show top 10 by AUC
    top10 = df_comp.sort_values("ADHD_AUC_Mean", ascending=False).head(10)
    print(top10.to_string(index=False))
else:
    print("❌ w2b_classical_vs_gnn.csv not found")

# ---------- 3. All architectures (with correct column names) ----------
all_file = W2_ROOT / "06_results/workflow2b/gnn_all_architectures.csv"
if all_file.exists():
    df_all = pd.read_csv(all_file)
    print("\n🔧 ALL ARCHITECTURES (column names):")
    print(df_all.columns.tolist())
    # Try to sort by AUC if column exists
    auc_col = None
    for col in df_all.columns:
        if 'auc' in col.lower():
            auc_col = col
            break
    if auc_col:
        print(f"\n🔧 ALL ARCHITECTURES (top 5 by {auc_col}):")
        print(df_all.sort_values(auc_col, ascending=False).head(5).to_string(index=False))
    else:
        print("\n⚠️ No AUC column found; showing first 5 rows:")
        print(df_all.head(5).to_string(index=False))
else:
    print("❌ gnn_all_architectures.csv not found")

# ---------- 4. LOSO per-site results ----------
loso_file = W2_ROOT / "06_results/workflow2b/w2b_loso_results.csv"
if loso_file.exists():
    df_loso = pd.read_csv(loso_file)
    print("\n🏥 PER-SITE / PER-FOLD RESULTS (first 10 rows):")
    print(df_loso.head(10).to_string(index=False))
else:
    print("❌ w2b_loso_results.csv not found")

# ---------- 5. Lambda sweep ----------
lambda_file = W2_ROOT / "06_results/workflow2c/w2c_lambda_performance.csv"
if lambda_file.exists():
    df_lambda = pd.read_csv(lambda_file)
    print("\n⚖️ LAMBDA SWEEP (site-invariance vs ADHD accuracy):")
    # Show best 5 by ADHD AUC
    if 'mean_adhd_auc' in df_lambda.columns:
        print(df_lambda.sort_values("mean_adhd_auc", ascending=False).head(5).to_string(index=False))
    else:
        print(df_lambda.head(5).to_string(index=False))
else:
    print("❌ w2c_lambda_performance.csv not found")

# ---------- 6. Harmonisation gains ----------
harmon_file = W2_ROOT / "06_results/workflow2a/w2_harmonization_gains.csv"
if harmon_file.exists():
    df_harm = pd.read_csv(harmon_file)
    print("\n🔄 HARMONISATION GAINS (site effect reduction):")
    print(df_harm.to_string(index=False))
else:
    print("❌ w2_harmonization_gains.csv not found")

# ---------- 7. Biological signal preservation ----------
bio_file = W2_ROOT / "06_results/workflow2a/w2_biological_signal_preservation.csv"
if bio_file.exists():
    df_bio = pd.read_csv(bio_file)
    print("\n🧬 BIOLOGICAL SIGNAL PRESERVATION (ADHD vs Control):")
    print(df_bio.to_string(index=False))
else:
    print("❌ w2_biological_signal_preservation.csv not found")

# ---------- 8. Final JSON summary ----------
json_file = W2_ROOT / "06_results/workflow2b/w2b_final_summary.json"
if json_file.exists():
    with open(json_file) as f:
        summary = json.load(f)
    print("\n📦 FINAL JSON SUMMARY:")
    print(json.dumps(summary, indent=2))
else:
    print("❌ w2b_final_summary.json not found")

print("\n" + "=" * 80)
print("✅ Done.")

WORKFLOW 2 – MAIN RESULTS (FIXED)

📊 GNN SUMMARY (BEST PER ARCHITECTURE):
Architecture  Mean_AUC  Std_AUC   CI_Low  CI_High  Mean_BA   Std_BA  N_Folds
         GCN  0.546838 0.073345 0.479005 0.614670 0.540651 0.038741        7
         GAT  0.575191 0.046604 0.532090 0.618292 0.549030 0.038374        7
        SAGE  0.550201 0.044796 0.508772 0.591630 0.514743 0.036070        7
         GIN  0.543739 0.063314 0.485183 0.602294 0.531283 0.057947        7

📈 CLASSICAL vs GNN (AUC / Balanced Acc):
     Representation Model  ADHD_AUC_Mean    CI_Lo    CI_Hi
  ComBat GraphPheno   SVM       0.659582 0.607499 0.713391
  ComBat GraphPheno    ET       0.648562 0.618623 0.687351
     Raw GraphPheno   SVM       0.648399 0.577900 0.713320
  ComBat GraphPheno    LR       0.643448 0.592308 0.693309
     Raw GraphPheno    LR       0.622432 0.559996 0.683724
ComBat FCGraphPheno    RF       0.620957 0.549725 0.682497
     ComBat FCPheno   SVM       0.612489 0.556517 0.659071
ComBat FCGraphPheno   SVM  